
# Data Access Connection using APP

In [0]:
service_credential = dbutils.secrets.get(scope="<secret-scope>",key="<service-credential-key>")

spark.conf.set("fs.azure.account.auth.type.<storage-account>.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.<storage-account>.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.<storage-account>.dfs.core.windows.net", "<application-id>")
spark.conf.set("fs.azure.account.oauth2.client.secret.<storage-account>.dfs.core.windows.net", service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.<storage-account>.dfs.core.windows.net", "https://login.microsoftonline.com/<directory-id>/oauth2/token")

### DATA Loading - Read All data

In [0]:
base_path = "abfss://bronze@awstoragedatalakeh.dfs.core.windows.net/"

# Get all folders inside bronze
folders = [f.path for f in dbutils.fs.ls(base_path) if f.isDir()]

for folder in folders:
    # Get the CSV file(s) inside the folder
    files = dbutils.fs.ls(folder)
    csv_files = [f.path for f in files if f.path.endswith(".csv")]
    
    if csv_files:
        csv_path = csv_files[0]  # Take the first CSV (assuming one per folder)
        
        # Clean the file name (remove extension for variable name)
        name = csv_path.split("/")[-1].replace(".csv", "")
        
        # Create DataFrame
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "true") \
            .load(csv_path)
        
        # Save DataFrame to variable df_<name>
        globals()[f"df_{name}"] = df


In [0]:
from pyspark.sql import DataFrame

base_path = "abfss://bronze@awstoragedatalakeh.dfs.core.windows.net/"

# Sales folders to merge
folders = ["AdventureWorks_Sales_2015", "AdventureWorks_Sales_2016", "AdventureWorks_Sales_2017"]

# Start with empty DataFrame
df_Sales_All: DataFrame = None

# Read and union all three sales files
for folder in folders:
    csv_path = f"{base_path}{folder}/*.csv"
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(csv_path)
    
    if df_Sales_All is None:
        df_Sales_All = df
    else:
        df_Sales_All = df_Sales_All.unionByName(df)

# Temp output path
temp_output = f"{base_path}AdventureWorks_Sales/tmp"

# Write to one file
df_Sales_All.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(temp_output)

# Find the part file and rename to AdventureWorks_Sales.csv
files = dbutils.fs.ls(temp_output)
part_file = [f.path for f in files if f.name.startswith("part-")][0]

final_output = f"{base_path}AdventureWorks_Sales/AdventureWorks_Sales.csv"

# Move and overwrite if exists
dbutils.fs.mv(part_file, final_output, True)

# Clean up the tmp folder
dbutils.fs.rm(temp_output, recurse=True)

print(f"Merged file created at: {final_output}")


# SILVER LAYER_SCRIPT Transformation

In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *

# Calender 

In [0]:

df_AdventureWorks_Calendar= df_AdventureWorks_Calendar.withColumn('Month',month(col('Date')))\
    .withColumn('Year',year(col('Date')))
df_AdventureWorks_Calendar.display ()


In [0]:
df_AdventureWorks_Calendar.write.format('parquet')\
    .mode('append')\
    .option("path","abfss://silver@awstoragedatalakeh.dfs.core.windows.net/AdventureWorks_Calendar")\
    .save()

# Customers

In [0]:
df_AdventureWorks_Customers.display()

In [0]:
df_AdventureWorks_Customers = df_AdventureWorks_Customers.withColumn("full name", concat_ws(' ', col("Prefix"), col("FirstName"), col("LastName")))    


In [0]:
df_AdventureWorks_Customers.display()

In [0]:
df_AdventureWorks_Customers.write.format('parquet')\
    .mode('append')\
    .option("path","abfss://silver@awstoragedatalakeh.dfs.core.windows.net/AdventureWorks_Customers")\
    .save()

# Product  Sub Categories

In [0]:
df_AdventureWorks_Product_Subcategories.display()

In [0]:
df_AdventureWorks_Product_Subcategories.write.format('parquet')\
    .mode('append')\
    .option("path","abfss://silver@awstoragedatalakeh.dfs.core.windows.net/AdventureWorks_Product_Subcategories")\
    .save()

# Products

In [0]:
df_AdventureWorks_Products.display()

In [0]:
 df_AdventureWorks_Products = df_AdventureWorks_Products.withColumn('ProductSKU',split(col('ProductSKU'),'-')[0])\
     .withColumn('ProductName',split(col('ProductName'),' ')[0])\
     .withColumn('ProductCost',round(col('ProductCost'),2))\
     .withColumn('ProductPrice',round(col('ProductPrice'),2))

In [0]:
df_AdventureWorks_Products.display()

In [0]:
df_AdventureWorks_Products.write.format('parquet')\
    .mode('append')\
    .option("path","abfss://silver@awstoragedatalakeh.dfs.core.windows.net/AdventureWorks_Products")\
    .save()

# Returns

In [0]:
df_AdventureWorks_Returns.display()

In [0]:
df_AdventureWorks_Returns.write.format('parquet')\
    .mode('append')\
    .option("path","abfss://silver@awstoragedatalakeh.dfs.core.windows.net/AdventureWorks_Returns")\
    .save()

In [0]:
df_AdventureWorks_Territories.display()

In [0]:
df_AdventureWorks_Territories.write.format('parquet')\
    .mode('append')\
    .option("path","abfss://silver@awstoragedatalakeh.dfs.core.windows.net/AdventureWorks_Territories")\
    .save()

# Sales

In [0]:
df_AdventureWorks_Sales_2015.display()

In [0]:
df_Sales_All.display()

In [0]:
df_Sales_All= df_Sales_All.withColumn('StockDate',to_timestamp(col('StockDate')))

In [0]:
df_Sales_All= df_Sales_All.withColumn('OrderNumber',regexp_replace(col('OrderNumber'),'S','T'))

In [0]:
df_Sales_All= df_Sales_All.withColumn('multiply',col('OrderQuantity')*col('OrderLineItem'))

In [0]:
df_Sales_All.display()

In [0]:
df_Sales_All.write.format('parquet')\
    .mode('append')\
    .option("path","abfss://silver@awstoragedatalakeh.dfs.core.windows.net/AdventureWorks_Sales")\
    .save()

# Product Categories

In [0]:
df_AdventureWorks_Product_Categories.display()
df_AdventureWorks_Product_Categories.write.format('parquet')\
    .mode('append')\
    .option("path","abfss://silver@awstoragedatalakeh.dfs.core.windows.net/AdventureWorks_Product_Categories")\
    .save()


# Sample Aggregation Testing


## Sales_Trend

In [0]:
df_Sales_All.groupby('OrderDate').agg(count('OrderNumber').alias('Total Orders')).display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_AdventureWorks_Customers.display()

In [0]:
df_joined = df_Sales_All.join(df_AdventureWorks_Customers,df_Sales_All.CustomerKey == df_AdventureWorks_Customers.CustomerKey,'inner').drop(df_AdventureWorks_Customers.CustomerKey)

In [0]:
df_joined.display()

## Maximum Order by top 20 Customer

In [0]:


df_order_per_customer = (
    df_joined
    .groupBy("CustomerKey", "full name")
    .agg(count("OrderNumber").alias("TotalOrders"))
    .orderBy(col("TotalOrders").desc())
    .limit(20)
)

df_order_per_customer.display()


Databricks visualization. Run in Databricks to view.

## Orders By Gender

In [0]:


df_orders_by_gender = (
    df_joined
    .groupBy("Gender") 
    .agg(count("OrderNumber").alias("TotalOrders"))
    .orderBy(col("TotalOrders").desc())
)

df_orders_by_gender.display()


Databricks visualization. Run in Databricks to view.

In [0]:
df_AdventureWorks_Sales_2015.display()

In [0]:
df_Sales_All.printSchema()


In [0]:
df_AdventureWorks_Products.columns

In [0]:
df_AdventureWorks_Product_Categories.display()
df_AdventureWorks_Product_Subcategories.display()


## Top 10 Customer by Spend

In [0]:
from pyspark.sql.functions import col, sum

# Join Sales + Products and compute SalesAmount
df_sales_with_price = (
    df_Sales_All
    .join(df_AdventureWorks_Products.select("ProductKey", "ProductPrice"), on="ProductKey", how="inner")
    .withColumn("SalesAmount", col("OrderQuantity") * col("ProductPrice"))
)

# Join with Customers and rename "full name" -> "FullName"
df_joined = df_sales_with_price.join(
    df_AdventureWorks_Customers.select(col("CustomerKey"), col("full name").alias("FullName")),
    on="CustomerKey",
    how="inner"
)

# Top 10 Customers by Spend
df_top_customers_by_spend = (
    df_joined
    .groupBy("CustomerKey", "FullName")
    .agg(sum("SalesAmount").alias("TotalSpend"))
    .orderBy(col("TotalSpend").desc())
    .limit(10)
)

df_top_customers_by_spend.display()


Databricks visualization. Run in Databricks to view.